# 01 · Carga, traducción y limpieza

**Etapa 1 del pipeline:** recolección y traducción automática de las reseñas al español.

- La traducción guarda su avance en disco cada 20 reseñas.
- La sección 2.1 hace una prueba corta y **estima el tiempo total** antes de lanzar la traducción completa.

Las funciones de traducción están en `src/pos_tagging.py`, para mantener el notebook legible y poder reutilizarlas.

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd

# Raíz del proyecto: el notebook se ejecuta desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pos_tagging as pt

RUTA_MUESTRA = RAIZ / "data" / "raw" / "muestra_en.csv"
DIR_PROCESADOS = RAIZ / "data" / "processed"
RUTA_PARCIAL = DIR_PROCESADOS / "resenas_es_parcial.csv"
RUTA_SALIDA = DIR_PROCESADOS / "resenas_es.csv"
DIR_PROCESADOS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 120)
print("Raíz del proyecto:", RAIZ)

Raíz del proyecto: C:\Users\ale23\Desktop\CUC\REPOS-CUC\recennas-cine\res_pel


## 1. Carga de la muestra

Columnas principales:

| Columna | Descripción |
|---|---|
| `id` | Identificador de la reseña dentro del proyecto |
| `movie_id`, `user_id` | Identificadores de IMDb de la película y del usuario |
| `fecha`, `anio` | Fecha de la reseña y su año |
| `calificacion` | Puntuación de 1 a 10 (eje B) |
| `spoiler` | Si la reseña contiene spoilers (eje D) |
| `resumen_en`, `texto_en` | Título y texto de la reseña en inglés |
| `n_palabras` | Número de palabras del texto en inglés |
| `genero`, `release_date` | Género asignado y fecha de estreno de la película |
| `periodo` | Periodo del análisis temporal (eje C) |

In [2]:
df = pd.read_csv(RUTA_MUESTRA)
print(f"{len(df)} reseñas, {df.shape[1]} columnas")
df.head(3)

3000 reseñas, 13 columnas


,id,movie_id,user_id,fecha,calificacion,spoiler,resumen_en,texto_en,n_palabras,genero,release_date,anio,periodo
0,1,tt0137523,ur21113658,20 November 2009,10,False,"The ultimate, perfect manly film","The Good: The comedy, Drama, Psychology, Philosophy, Acting, stellar direction, great characters, gritty violence Th...",178,Drama,1999-10-15,2009,1998-2009
1,2,tt0472033,ur12568341,19 July 2011,1,True,Poor story and character development.,"The only real reason people like this movie so much is that Tim Burton slapped his name on it at last-minute as a ""g...",237,Animation,2009-09-09,2011,2010-2018
2,3,tt0117666,ur0230283,3 February 1999,10,False,"I cried, and I don't cry much.","The film just blew my mind. By the time B.B. Thornton is done with his little career, his photo will be located in t...",102,Drama,1997-03-14,1999,1998-2009


### 1.1 Verificación de los requisitos del corpus (sección 4.3)

Antes de invertir horas en la traducción se comprueba que la muestra cumple los mínimos del enunciado. Si alguna verificación falla, la celda se detiene con un error.

In [3]:
por_genero = df["genero"].value_counts()
por_periodo = df["periodo"].value_counts()

print("Reseñas por género:", por_genero.to_dict())
print("Reseñas por periodo:", por_periodo.to_dict())
print(f"Longitud (palabras): mín {df['n_palabras'].min()}, máx {df['n_palabras'].max()}")

assert len(df) >= 3000, "Se necesitan al menos 3.000 reseñas"
assert len(por_genero) >= 5 and por_genero.min() >= 400, "5 géneros con al menos 400 reseñas cada uno"
assert len(por_periodo) >= 2 and por_periodo.min() >= 1000, "2 periodos con al menos 1.000 reseñas cada uno"
assert df["n_palabras"].between(30, 400).all(), "Cada reseña debe tener entre 30 y 400 palabras"
assert df["id"].is_unique, "Los id de reseña deben ser únicos"
print("\nLa muestra cumple los requisitos de la sección 4.3.")

Reseñas por género: {'Drama': 600, 'Animation': 600, 'Action': 600, 'Horror': 600, 'Comedy': 600}
Reseñas por periodo: {'1998-2009': 1500, '2010-2018': 1500}
Longitud (palabras): mín 31, máx 400

La muestra cumple los requisitos de la sección 4.3.


## 2. Traducción automática al español

**Modelo:** [`Helsinki-NLP/opus-mt-en-es`](https://huggingface.co/Helsinki-NLP/opus-mt-en-es) (MarianMT). Es gratuito, se ejecuta localmente y no requiere claves de API. La primera vez se descarga (~300 MB) y queda guardado en la caché de Hugging Face.

**Cómo se traduce cada reseña** (funciones en `src/pos_tagging.py`):

1. **Preparación** (`preparar_texto_en`): se quitan etiquetas HTML como `<br />` y se separan oraciones que en IMDb suelen venir pegadas (`"impression.This film"` → `"impression. This film"`). No se cambian mayúsculas, tildes ni puntuación.
2. **División en oraciones** (`dividir_oraciones`): con `nltk.sent_tokenize`. El modelo trunca entradas de más de 256 tokens, así que traducir la reseña entera haría perder texto. Si una "oración" supera las 100 palabras (reseñas sin puntuación), se corta en trozos.
3. **Traducción por lotes** (`traducir_oraciones`): las oraciones de varias reseñas se ordenan por longitud y se traducen en lotes de 16. Así cada lote tiene oraciones de tamaño parecido y se desperdicia menos cómputo en relleno.
4. **Reensamblado** (`traducir_resenas`): las oraciones traducidas se vuelven a unir en el orden original de cada reseña.

Las traducciones **no se corrigen a mano**: se analizan tal como las produce el modelo y sus errores se discuten más adelante (eje D y comparación NLTK vs spaCy).

In [4]:
%pip install sentencepiece sacremoses
# tuve que ejcutar esto ya que el siguiente bloque daba problemas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import torch

pt.asegurar_recursos_nltk()
traductor = pt.cargar_traductor()

print("Dispositivo:", traductor["dispositivo"])
print("Hilos de CPU usados por torch:", torch.get_num_threads())

C:\Users\ale23\Desktop\CUC\REPOS-CUC\ProyectoExploracion\MineriaDatos\src\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ale23\Desktop\CUC\REPOS-CUC\ProyectoExploracion\MineriaDatos\src\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ale23\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to 

Dispositivo: cpu
Hilos de CPU usados por torch: 10


### 2.1 Prueba rápida y estimación del tiempo total

Se traducen 5 reseñas para comprobar que todo funciona y para estimar cuánto tardará el corpus completo. La estimación se basa en palabras por segundo, porque el tiempo de traducción depende sobre todo de la longitud del texto.


In [6]:
LOTE = 16
NUM_BEAMS = None  # None = valor del modelo (4)

prueba = df.sample(5, random_state=42)
inicio = time.time()
traducciones = pt.traducir_resenas(prueba["texto_en"].tolist(), traductor,
                                   lote=LOTE, num_beams=NUM_BEAMS)
segundos = time.time() - inicio

palabras_por_seg = prueba["n_palabras"].sum() / segundos
estimado_h = df["n_palabras"].sum() / palabras_por_seg / 3600
print(f"Prueba: {prueba['n_palabras'].sum()} palabras en {segundos:.1f} s "
      f"({palabras_por_seg:.1f} palabras/s)")
print(f"Tiempo estimado para todo el corpus: {estimado_h:.1f} horas")

Prueba: 753 palabras en 22.7 s (33.2 palabras/s)
Tiempo estimado para todo el corpus: 4.5 horas


In [12]:
# Comparación lado a lado de la prueba (solo los primeros 400 caracteres)
for (_, fila), texto_es in zip(prueba.iterrows(), traducciones):
    print(f"── Reseña {fila['id']} · {fila['genero']} · {fila['calificacion']}/10")
    print("EN:", fila["texto_en"][:3000])
    print("ES:", texto_es[:3000], "\n")

── Reseña 1802 · Horror · 2/10
EN: First off I'd like to say that I'm not a huge fan of the "The Green Mile" and whilst I like "The Shawshank Redemption" I personally think the film is overrated. That being said I can understand and appreciate why people would like or even love those movies.I can not however, understand why people here are giving Frank Darabont's newest movie such good reviews or why its holds the rating of 7.5 which it clearly doesn't deserve. This movie is truly awful, one of the worst that I have seen this year.The characters have barely any substance to them and are not endearing what so ever. The plot drags on in a mundane and pointless manner and the acting is spotty at best.Whilst this movie steals moments from great films (most notably "Aliens") and a great music score from the "300", it completely fails to capture the same (or in fact any) sense of urgency, tension, awe and perhaps most importantly in a horror movie; fear.This may in part be due to the charact

### 2.2 Traducción del corpus completo (reanudable)


In [13]:
traducido = pt.traducir_corpus(df, traductor, RUTA_PARCIAL,
                               resenas_por_bloque=20,
                               lote=LOTE, num_beams=NUM_BEAMS)
print(f"Reseñas traducidas: {traducido['id'].nunique()} de {len(df)}")

100%|██████████| 3000/3000 [5:57:41<00:00,  7.15s/reseña]  

3000 reseñas traducidas en 357.7 min
Reseñas traducidas: 3000 de 3000


### 2.3 Unión y guardado de `resenas_es.csv`

Se une la traducción con la muestra original. El CSV final **conserva el texto en inglés (`texto_en`) junto al texto en español (`texto_es`)**, como pide el enunciado: así se pueden revisar traducciones dudosas y responder las preguntas sobre el efecto de la traducción en el etiquetado.

Si una interrupción dejó alguna reseña repetida en el archivo parcial, se conserva solo una copia.

In [14]:
traducido = traducido.drop_duplicates(subset="id", keep="last")
faltantes = set(df["id"]) - set(traducido["id"])
assert not faltantes, (f"Faltan {len(faltantes)} reseñas por traducir: "
                       "vuelve a ejecutar la celda de la sección 2.2")

resenas_es = df.merge(traducido, on="id", how="left")

# texto_es se coloca justo después de texto_en para compararlos fácilmente
columnas = list(df.columns)
columnas.insert(columnas.index("texto_en") + 1, "texto_es")
resenas_es = resenas_es[columnas]

resenas_es.to_csv(RUTA_SALIDA, index=False, encoding="utf-8")
print(f"Guardado {RUTA_SALIDA.relative_to(RAIZ)}: {resenas_es.shape[0]} filas, "
      f"{resenas_es.shape[1]} columnas")
resenas_es[["id", "genero", "calificacion", "texto_en", "texto_es"]].head()

Guardado data\processed\resenas_es.csv: 3000 filas, 14 columnas


,id,genero,calificacion,texto_en,texto_es
0,1,Drama,10,"The Good: The comedy, Drama, Psychology, Philosophy, Acting, stellar direction, great characters, gritty violence Th...","Lo bueno: La comedia, Drama, Psicología, Filosofía, Actuar, dirección estelar, grandes personajes, violencia arenosa..."
1,2,Animation,1,"The only real reason people like this movie so much is that Tim Burton slapped his name on it at last-minute as a ""g...",La única razón real por la que a la gente le gusta tanto esta película es que Tim Burton abofeteó su nombre en el úl...
2,3,Drama,10,"The film just blew my mind. By the time B.B. Thornton is done with his little career, his photo will be located in t...","La película me voló la cabeza. Para cuando B. B. Thornton se hace con su pequeña carrera, su foto se ubicará en el d..."
3,4,Action,4,"For me, M:i:III was an impulse watch, something I went to see and paid the hefty admission for because it was there....","Para mí, M:i:III era un reloj de impulso, algo que fui a ver y pagué la admisión fuerte porque estaba allí. Desafort..."
4,5,Horror,4,"PLOT SPOILERS: I guess I should have seen it coming, what with the sixth sense it's almost like a new trend in plot ...","Supongo que debería haberlo visto venir, con el sexto sentido es casi como una nueva tendencia en giros de trama. Pe..."


### 2.4 Revisión rápida de la traducción

Una comprobación inicial, antes de la limpieza: traducciones vacías y relación de longitud entre el español y el inglés. El español suele ser algo más largo que el inglés (ratio cercano a 1,1–1,2). Un ratio muy bajo puede indicar texto perdido y uno muy alto, repeticiones del modelo. La detección de traducciones fallidas (por ejemplo, texto que quedó en inglés) se hace en la limpieza.

In [16]:
resenas_es["n_palabras_es"] = resenas_es["texto_es"].fillna("").str.split().str.len()
ratio = resenas_es["n_palabras_es"] / resenas_es["n_palabras"]

print("Traducciones vacías:", (resenas_es["n_palabras_es"] == 0).sum())
print(ratio.describe().round(2))


Traducciones vacías: 0
count    3000.00
mean        1.04
std         0.11
min         0.77
25%         1.00
50%         1.04
75%         1.07
max         5.02
dtype: float64

Reseñas con ratio < 0,7 (posible texto perdido): 0
Reseñas con ratio > 1,6 (posibles repeticiones): 3
